# AIDA 1141 Final Project – 04 Errors, Groups, Ties and the Review Policy

**Data version 2** (`shared/synthetic-data/`), using notebook 02's saved validation predictions. Nothing is refit or re-tuned.

Notebook 03 found that neither the days-until-start rule nor the logistic regression consistently does better on validation. This notebook finishes the validation work before the single test evaluation:

1. revisits how **tied scores** are handled in the matched-volume comparison. That comparison split a group of three equal-scoring assessments; the threshold-based flagging policies kept the group together;
2. explains what a flag means in a coordinator's terms, including the workload of flagging about 47% of early-warning assessments;
3. reviews the **missed at-risk assessments** and the **flags on assessments that were not at risk**, using only information available at each assessment;
4. reports results **by group**, with their sizes, and reviews bias and privacy;
5. applies development criteria fixed before this notebook's comparison, then **chooses and freezes the review policy**.

Everything uses **validation** data. The test set and the time-based check stay unopened until the policy is frozen. The simulation's evaluation-only values are not used. All records are synthetic, and the rates describe the generator's assumptions, not real employees or real HR performance.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")

settings = json.loads((OUTPUT_DIR / "model_settings.json").read_text())          # from notebook 02
predictions = pd.read_csv(OUTPUT_DIR / "validation_predictions.csv")             # from notebook 02
read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
assessments = read(SD / "requests/data/assessments.csv")
requests = read(SD / "cleaning/output/requests_cleaned.csv")
employee_split = read(OUTPUT_DIR / "employee_split.csv")

sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
SOURCES = {"requests_cleaned.csv": SD / "cleaning/output/requests_cleaned.csv",
           "assessments.csv": SD / "requests/data/assessments.csv",
           "employee_split.csv": OUTPUT_DIR / "employee_split.csv", "feature_list.csv": OUTPUT_DIR / "feature_list.csv"}
assert {n: sha(p) for n, p in SOURCES.items()} == settings["source_fingerprints_sha256"], \
    "The inputs don't match the fingerprints saved with the model in notebook 02"

# Eligible validation assessments rebuilt from the source files with notebook 01's rules
source = assessments.merge(employee_split[["employee_id", "split"]], on="employee_id", how="left", validate="many_to_one")
eligible_ids = set(source.loc[(source["split"] == "validation") & (source["at_risk"] != "")
                              & (source["past_deadline_at_assessment"] == "False"), "assessment_id"])
assert set(predictions["assessment_id"]) == eligible_ids and predictions["assessment_id"].is_unique, \
    "The predictions are not exactly the eligible validation assessments"

# The saved rows must also carry the source's task, employee and outcome label (label normalised to 0/1)
saved = predictions.set_index("assessment_id")[["request_id", "employee_id", "at_risk"]]
truth = source.set_index("assessment_id").loc[saved.index, ["request_id", "employee_id", "at_risk"]]
assert truth["at_risk"].isin(["True", "False"]).all() and saved["at_risk"].isin([0, 1]).all(), "Unexpected label values"
assert (saved["request_id"] == truth["request_id"]).all() and (saved["employee_id"] == truth["employee_id"]).all(), \
    "Saved task or employee IDs don't match the source assessments"
assert (saved["at_risk"] == truth["at_risk"].map({"True": 1, "False": 0})).all(), \
    "Saved outcome labels don't match the source assessments"

# Add the remaining model inputs (observable at each assessment) for the error and group review
FEATURES = settings["features"]
v = predictions.merge(assessments[["assessment_id", "current_status", "business_days_in_current_status", "team_open_workload"]],
                      on="assessment_id", how="left", validate="one_to_one")
v = v.merge(requests[["request_id", "hiring_department", "intake_channel", "info_issue_at_intake"]],
            on="request_id", how="left", validate="many_to_one")
v["intake_channel"] = v["intake_channel"].replace("", "unknown")              # as in notebook 02
for c in ["business_days_in_current_status", "team_open_workload"]:
    v[c] = v[c].astype(int)
v["y"] = v["at_risk"].astype(int)
assert len(v) == len(predictions) and v[FEATURES].notna().all().all(), "Some model inputs are missing after the merge"
assert not (v[FEATURES].astype(str).apply(lambda c: c.str.strip()) == "").any().any(), "Some model inputs are blank"

DAYS_THRESHOLD = settings["days_threshold"]
MODEL_THRESHOLD = settings["model_threshold"]
assert (v["days_rule"] == (v["business_days_until_start"] <= DAYS_THRESHOLD)).all()
assert (v["logistic_regression"] == (v["risk_probability"] >= MODEL_THRESHOLD)).all()
METHODS = {"days rule": "days_rule", "logistic regression": "logistic_regression"}

print(f"Data version {DATA_VERSION}: {len(v):,} validation early-warning assessments, {v['y'].sum()} at risk, "
      f"{v.loc[v['y'] == 1, 'request_id'].nunique()} at-risk tasks, {v['employee_id'].nunique()} employees")
print(f"Training-selected thresholds from notebook 02: days until start <= {DAYS_THRESHOLD}; "
      f"model risk >= {MODEL_THRESHOLD:.4f}")

Data version 2: 1,033 validation early-warning assessments, 194 at risk, 154 at-risk tasks, 209 employees
Training-selected thresholds from notebook 02: days until start <= 8; model risk >= 0.1338


## A. Shared measures

The same measures are used throughout, at two levels:

- **Assessment recall:** of the at-risk assessments, how many received a warning?
- **Task recall:** of the at-risk tasks, how many received at least one early-warning flag? A task can be assessed twice (checkpoints 1 and 2), so a task counts as flagged if any of its assessments was flagged. This is **task deduplication**; in a live queue, a later assessment of a task already on the list would update its entry rather than create a second one.
- **Precision:** of the flagged assessments, how many were labelled at risk? Its denominator is the number flagged.
- **Flags on not-at-risk assessments:** flags on assessments labelled not at risk (the task was finished in time). They count toward review volume, but they do not tell us whether a check would have been useful or how much staff time it would have taken.
- **Unique tasks flagged:** the number of different tasks flagged across the observed checkpoints. It shows how much the assessment counts shrink after deduplication; it does not measure the size of a live queue at any one time.

Counts are whole numbers; rates and percentages may have decimals. The function below only accepts real True/False (or 0/1) flags. The fractional expected results for tied orders in section B are calculated separately: they are analytical estimates, not sets of flags.

In [2]:
def summarize(flag, rows=None):
    # Review volume and outcomes when the given assessments are flagged
    rows = v if rows is None else rows
    if isinstance(flag, pd.Series) and not flag.index.equals(rows.index):
        raise ValueError("Flags must align with the assessment rows")
    flag = pd.Series(flag, index=rows.index)
    if not flag.isin([True, False]).all():                 # rejects missing values and strings such as "False"
        raise ValueError("Flags must contain only booleans or 0/1, with no missing values")
    flag = flag.astype(bool)
    y = rows["y"] == 1
    tp, fp, fn = int((flag & y).sum()), int((flag & ~y).sum()), int((~flag & y).sum())
    tasks = rows.assign(flag=flag).groupby("request_id").agg(at_risk=("y", "max"), flagged=("flag", "any"))
    at_risk_tasks = int(tasks["at_risk"].sum())
    caught_tasks = int((tasks["at_risk"].astype(bool) & tasks["flagged"]).sum())
    return {"assessments": len(rows), "at_risk": int(y.sum()), "flagged": int(flag.sum()),
            "at_risk_flagged": tp, "not_at_risk_flagged": fp, "at_risk_missed": fn,
            "assessment_recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "unique_tasks_flagged": int(tasks["flagged"].sum()), "at_risk_tasks": at_risk_tasks,
            "at_risk_tasks_flagged": caught_tasks, "at_risk_tasks_never_flagged": at_risk_tasks - caught_tasks,
            "task_recall": caught_tasks / at_risk_tasks if at_risk_tasks else np.nan}


COUNT_COLS = ["assessments", "at_risk", "flagged", "at_risk_flagged", "not_at_risk_flagged", "at_risk_missed",
              "unique_tasks_flagged", "at_risk_tasks", "at_risk_tasks_flagged", "at_risk_tasks_never_flagged"]
overall = pd.DataFrame({name: summarize(v[col]) for name, col in METHODS.items()}).T
overall[COUNT_COLS] = overall[COUNT_COLS].astype(int)
overall = overall.astype(object)                       # keep counts as whole numbers when rows are read back
display(overall)

,assessments,at_risk,flagged,at_risk_flagged,not_at_risk_flagged,at_risk_missed,assessment_recall,precision,unique_tasks_flagged,at_risk_tasks,at_risk_tasks_flagged,at_risk_tasks_never_flagged,task_recall
days rule,1033,194,486,177,309,17,0.912371,0.364198,366,154,154,0,1.0
logistic regression,1033,194,494,178,316,16,0.917526,0.360324,368,154,154,0,1.0


## B. Tied scores

**What happened in notebook 03.** To compare the methods at the same review volume, the model selected its N highest-risk assessments for each rule threshold. At N = 486 (the rule's volume at 8 days), the cutoff fell inside a group of three assessments with exactly the same score. An assessment-ID tie-breaker put two of them on the list and left one off.

**Why it matters.** The ID tie-breaker is repeatable, but it gave different decisions to cases the model could not tell apart, for a reason with no operational meaning. That is a **consistency** concern. It is not evidence of discrimination against a protected group. Different inputs can also produce equal scores, so the rule adopted here is about scores: **equal scores receive equal flag decisions.**

**Rules adopted:**

- **Real flagging policies use cutoffs** (days ≤ threshold, or risk ≥ threshold). A cutoff always gives equal scores the same decision, so it never splits a tie.
- **Comparisons at a fixed review volume report the expected result across every order of the tied cases.** This is an analytical average: it can include a fraction of an assessment, so it describes the comparison, not a queue anyone could run.
- **Including or excluding the whole tied group** is reported alongside, as a sensitivity check.

The cell below checks ties at full precision, compares the four ways of handling the tie at every matched volume, and confirms that both real policies give equal scores equal decisions.

In [3]:
risk, days = v["risk_probability"], v["business_days_until_start"]

# Ties among the model's scores, at full precision
score_counts = risk.value_counts()
tied_scores = score_counts[score_counts > 1]
print(f"Model scores: {risk.nunique():,} distinct values among {len(v):,} assessments; "
      f"{len(tied_scores)} tied groups (largest {tied_scores.max() if len(tied_scores) else 1})")

by_model = v.sort_values(["risk_probability", "assessment_id"], ascending=[False, True])


def top_n_with_ties(n):
    # The model's N highest-risk assessments under four ways of handling a tie at the cutoff
    if not 0 <= n <= len(v):
        raise ValueError("Review volume must be between zero and all assessments")
    if n == 0:
        empty = summarize(pd.Series(False, index=v.index))
        return {"boundary": np.nan, "tied_group_size": 0, "tie_split": False, "id": empty.copy(),
                "include": empty.copy(), "exclude": empty.copy(), "expected_at_risk": 0.0}
    boundary = by_model["risk_probability"].iloc[n - 1]
    above, group = risk > boundary, risk == boundary
    slots = n - int(above.sum())                                  # places left for the tied group
    split = 0 < slots < int(group.sum())                          # the cutoff falls inside a tied group
    expected_at_risk = v.loc[above, "y"].sum() + v.loc[group, "y"].sum() * slots / int(group.sum())
    id_flag = v.index.isin(by_model.index[:n])
    result = {"boundary": boundary, "tied_group_size": int(group.sum()), "tie_split": split,
              "id": summarize(id_flag), "include": summarize(above | group),
              "exclude": summarize(above if split else above | group), "expected_at_risk": expected_at_risk}
    # Without a split, all four ways must select the same N assessments
    assert result["id"]["flagged"] == n
    if not split:
        assert result["include"]["flagged"] == n and result["exclude"]["flagged"] == n
        assert np.isclose(result["expected_at_risk"], result["id"]["at_risk_flagged"])
    return result


rows = []
for k in range(2, 15):
    rule = summarize(days <= k)
    n = rule["flagged"]
    t = top_n_with_ties(n)
    rows.append({"rule_threshold": f"<= {k} days", "review_volume": n, "rule_at_risk_reached": rule["at_risk_flagged"],
                 "model_at_risk_reached_expected": t["expected_at_risk"],
                 "model_at_risk_reached_id_tiebreak": t["id"]["at_risk_flagged"],
                 "include_group_volume": t["include"]["flagged"], "include_group_at_risk": t["include"]["at_risk_flagged"],
                 "exclude_group_volume": t["exclude"]["flagged"], "exclude_group_at_risk": t["exclude"]["at_risk_flagged"],
                 "rule_at_risk_tasks_missed": rule["at_risk_tasks_never_flagged"],
                 "model_at_risk_tasks_missed_include": t["include"]["at_risk_tasks_never_flagged"],
                 "model_at_risk_tasks_missed_exclude": t["exclude"]["at_risk_tasks_never_flagged"],
                 "cutoff_tied_group_size": t["tied_group_size"], "tie_split_by_id": t["tie_split"]})
matched = pd.DataFrame(rows)
matched.insert(0, "data_version", DATA_VERSION)
matched.to_csv(OUTPUT_DIR / "matched_volume_ties_validation.csv", index=False)

# The tied group at the rule's volume: same score at full precision; are the inputs identical too?
N_RULE = int(v["days_rule"].sum())
boundary = top_n_with_ties(N_RULE)["boundary"]
group_486 = v[risk == boundary].sort_values("assessment_id")
group_486 = group_486.assign(position_in_model_order=[by_model.index.get_loc(i) + 1 for i in group_486.index],
                             selected_by_id_tiebreak=group_486.index.isin(by_model.index[:N_RULE]))
identical_inputs = bool((group_486[FEATURES].nunique() == 1).all())
print(f"At the rule's volume ({N_RULE}), the cutoff score {float(boundary)!r} is shared by {len(group_486)} assessments; "
      f"identical on all {len(FEATURES)} model inputs: {identical_inputs}")
display(group_486[["assessment_id", "employee_id", "position_in_model_order", "selected_by_id_tiebreak", "at_risk"] + FEATURES])

# Both real policies are cutoffs: equal scores must receive equal flag decisions
equal_days_equal_flag = bool((v.groupby("business_days_until_start")["days_rule"].nunique() == 1).all())
equal_risk_equal_flag = bool((v.groupby("risk_probability")["logistic_regression"].nunique() == 1).all())
print(f"Equal day counts receive equal rule decisions: {equal_days_equal_flag}; "
      f"equal model scores receive equal model decisions: {equal_risk_equal_flag}")
assert equal_days_equal_flag, "Equal day counts received different rule decisions"
assert equal_risk_equal_flag, "Equal model scores received different model decisions"
display(matched.drop(columns="data_version").round(2))

Model scores: 1,025 distinct values among 1,033 assessments; 7 tied groups (largest 3)
At the rule's volume (486), the cutoff score 0.1371989573653714 is shared by 3 assessments; identical on all 9 model inputs: True


,assessment_id,employee_id,position_in_model_order,selected_by_id_tiebreak,at_risk,task_type,hiring_department,employee_type,intake_channel,info_issue_at_intake,current_status,business_days_until_start,business_days_in_current_status,team_open_workload
199,ASM-00999,EMP-0754,485,True,1,system_access,Parks,seasonal,web_form,False,in_progress,5,1,32
913,ASM-04733,EMP-0003,486,True,0,system_access,Parks,seasonal,web_form,False,in_progress,5,1,32
914,ASM-04734,EMP-0462,487,False,0,system_access,Parks,seasonal,web_form,False,in_progress,5,1,32


Equal day counts receive equal rule decisions: True; equal model scores receive equal model decisions: True


,rule_threshold,review_volume,rule_at_risk_reached,model_at_risk_reached_expected,model_at_risk_reached_id_tiebreak,include_group_volume,include_group_at_risk,exclude_group_volume,exclude_group_at_risk,rule_at_risk_tasks_missed,model_at_risk_tasks_missed_include,model_at_risk_tasks_missed_exclude,cutoff_tied_group_size,tie_split_by_id
0,<= 2 days,33,33,33.00,33,33,33,33,33,121,121,121,1,False
1,<= 3 days,69,63,61.00,61,69,61,69,61,91,93,93,1,False
2,<= 4 days,111,91,87.00,87,111,87,111,87,63,67,67,1,False
3,<= 5 days,360,154,153.00,153,360,153,360,153,0,10,10,1,False
4,<= 6 days,411,168,165.00,165,411,165,411,165,0,4,4,1,False
5,<= 7 days,445,171,174.00,174,445,174,445,174,0,1,1,1,False
6,<= 8 days,486,177,177.67,178,487,178,484,177,0,0,0,3,True
7,<= 9 days,536,182,184.00,184,536,184,536,184,0,0,0,1,False
8,<= 10 days,568,188,188.00,188,568,188,568,188,0,0,0,1,False
9,<= 11 days,605,190,190.00,190,605,190,605,190,0,0,0,1,False


**Ties in the review order.** Ties also affect the order of a review list. Many assessments share the same day count (all checkpoint-2 assessments are at exactly five days), so "fewest days first" has large ties; the model's scores have a few small ones. As with matched volumes, a partial review of the list is compared using the **expected** number of at-risk assessments reached across every order of tied cases. If the whole list is reviewed, the order within ties makes no difference to what is reached.

The cell below does this for the rule's fixed list at 25% and 50% of the list, for the policy criteria in section H. These review budgets were carried forward from notebook 03 and fixed for this comparison. They are development choices, not independently selected evaluation points.

Review budgets are rounded to the nearest whole assessment (25% of 486 becomes 122; 50% becomes 243). Expected outcomes can still be fractional because they average over possible orders within a tied group.

In [4]:
LIST = v[v["days_rule"]]
FRACTIONS = [0.25, 0.50]


def expected_reached(priority, budget):
    # Expected at-risk assessments reached in the first `budget` places, averaged over every order within ties.
    # `priority`: higher value = reviewed earlier.
    assert priority.index.equals(LIST.index), "Priorities must align with the review list"
    assert np.isfinite(priority.to_numpy(dtype=float)).all(), "Priorities must be complete and finite"
    assert 0 <= budget <= len(LIST), "Review budget must be between zero and the list length"
    reached, used = 0.0, 0
    for _, group in sorted(LIST.groupby(priority), key=lambda kv: kv[0], reverse=True):
        take = min(len(group), budget - used)
        if take <= 0:
            break
        reached += group["y"].sum() * take / len(group)
        used += take
    return reached


ORDERS = {"fewest days first": -LIST["business_days_until_start"], "highest model risk first": LIST["risk_probability"]}
order_rows = []
for f in FRACTIONS:
    budget = int(round(f * len(LIST)))
    row = {"share_of_list_reviewed": f"{f:.0%}", "assessments_reviewed": budget}
    for name, priority in ORDERS.items():
        row[f"{name}: expected at-risk reached"] = expected_reached(priority, budget)
    order_rows.append(row)
order_ties = pd.DataFrame(order_rows)
order_ties["difference (model order minus days order)"] = (order_ties["highest model risk first: expected at-risk reached"]
                                                          - order_ties["fewest days first: expected at-risk reached"])
order_ties.insert(0, "data_version", DATA_VERSION)
order_ties.to_csv(OUTPUT_DIR / "review_order_ties_validation.csv", index=False)
print(f"Rule's fixed list: {len(LIST)} assessments, {LIST['y'].sum()} at risk. "
      f"Largest tie in day counts on the list: {LIST['business_days_until_start'].value_counts().max()} assessments")
display(order_ties.drop(columns="data_version").round(2))

Rule's fixed list: 486 assessments, 177 at risk. Largest tie in day counts on the list: 249 assessments


,share_of_list_reviewed,assessments_reviewed,fewest days first: expected at-risk reached,highest model risk first: expected at-risk reached,difference (model order minus days order)
0,25%,122,93.78,91.0,-2.78
1,50%,243,124.40,132.0,7.60
